# MRIxFields2026 · Task 3 · audited reproduction of `mc_ssim_slice_avg_tta`

Runs the full training chain of our submitted model (MIM pretraining → 25-epoch fine-tune → widen → 8-epoch multi-contrast/SSIM/slice fine-tune → average e6–e8) under the official audit tools, and keeps everything that matters on Drive:

* **checkpoints** → `MyDrive/mrixfields2026/docker_final_logs/runs/…`
* **audit logs** (`audit-logs/`), console logs, environment record → `MyDrive/mrixfields2026/docker_final_logs/…`

Data is read from `MyDrive/mrixfields/{training_prospective, training_retrospective, validation_prospective}` (original NIfTI). Slices are extracted once to the local disk (`/content/preprocessed`, ~50 GB).

**If the session disconnects, just re-run the training cell** — every stage resumes from its latest checkpoint on Drive. Use an A100 / L4 runtime if you can: on an A100 the whole chain is roughly 8–10 h (preprocessing ~1–2 h of that); a T4 takes several sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- paths (edit if yours differ) ------------------------------------------------------
DRIVE_DATA = "/content/drive/MyDrive/mrixfields"                       # training_prospective/, training_retrospective/, validation_prospective/
DRIVE_LOGS = "/content/drive/MyDrive/mrixfields2026/docker_final_logs"  # https://drive.google.com/drive/folders/17PG9dhKD4Uz6gwRvBSYZKtm9DDM3v0jI
REPO_URL   = "https://github.com/denizberkin/mrixfields-analysis.git"
GITHUB_TOKEN = ""                                                      # only if the repo is private
WORK         = "/content/work"
PREPROCESSED = "/content/preprocessed"                                 # local disk; rebuilt if missing

import os
os.makedirs(DRIVE_LOGS, exist_ok=True); os.makedirs(WORK, exist_ok=True)
assert os.path.isdir(DRIVE_DATA), DRIVE_DATA
print(sorted(os.listdir(DRIVE_DATA)))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1
!free -g | head -2

In [ ]:
# ---- code -------------------------------------------------------------------------------
url = REPO_URL if not GITHUB_TOKEN else REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
if not os.path.isdir(f"{WORK}/mrixfields-analysis"):
    !git clone --recurse-submodules "{url}" {WORK}/mrixfields-analysis
!cd {WORK}/mrixfields-analysis && git pull --recurse-submodules -q && git submodule update --init --recursive -q && git log --oneline -1 && git -C experiment-pipeline log --oneline -1
!pip install -q -r {WORK}/mrixfields-analysis/audit/requirements.txt
%cd {WORK}/mrixfields-analysis/experiment-pipeline
!ls audit_monitor.py audit_utils.py audit_dataset.py train_task3_audit.py

### Smoke test (≈10–15 min) — run once before the real thing
Two subjects per domain, two field strengths, 20 pretraining steps, 2 epochs per stage, into a separate `smoke/` folder. Checks the data path, the audit tools, Drive writes and resume without spending hours. Delete `DRIVE_LOGS/smoke` afterwards if you like.

In [ ]:
!python3 -u train_task3_audit.py --smoke --stages preprocess stage1 widen stage2 average export     --raw-dir "{DRIVE_DATA}" --preprocessed-dir "{PREPROCESSED}"     --output-dir "{DRIVE_LOGS}/smoke/runs" --mirror-dir "{DRIVE_LOGS}/smoke" ;  python3 audit_mirror.py --src audit-logs console-logs --dst "{DRIVE_LOGS}/smoke" --delete
!ls "{DRIVE_LOGS}/smoke/audit-logs" | head; tail -3 "{DRIVE_LOGS}"/smoke/audit-logs/training_*.log.csv

### Optional: fetch the shipped container files (weights + inference code) for the consistency check
Pulls `docker.synapse.org/syn76236366/task3:v2`'s application layers over the registry API (no Docker needed) into `DRIVE_LOGS/docker_submission/`. The export stage then writes `reference_comparison.json` against the shipped `task3.pt`, and the package includes the container code. Needs a Synapse personal access token; it is only exchanged for a short-lived registry token and never written anywhere.

In [ ]:
import getpass, os
os.environ["PERSONAL_ACCESS_TOKEN"] = getpass.getpass("Synapse personal access token: ")
!python3 {WORK}/mrixfields-analysis/scripts/pull_synapse_docker.py extract --layers 7 8 9 10 11 12 13     --include 'app/*' --out "{DRIVE_LOGS}/docker_submission" && rm -rf "{DRIVE_LOGS}/docker_submission/.layers"
del os.environ["PERSONAL_ACCESS_TOKEN"]
!sha256sum "{DRIVE_LOGS}/docker_submission/app/weights/task3.pt"   # expected be132195d8afc96494aa9c3a252065cadb97084547d421111b79dd392567063b

### The audited run — all stages (re-run this cell to resume after a disconnect)
`preprocess → stage1 → widen → stage2 → average → export → predict`. Each stage is skipped once its outputs exist; interrupted stages resume from their newest checkpoint on Drive. The second command copies the finalised (`.log.csv`) audit logs after the tools' exit-time rename.

In [ ]:
!python3 -u train_task3_audit.py --stages all     --raw-dir "{DRIVE_DATA}" --preprocessed-dir "{PREPROCESSED}"     --output-dir "{DRIVE_LOGS}/runs" --mirror-dir "{DRIVE_LOGS}"     --reference-weights "{DRIVE_LOGS}/docker_submission/app/weights/task3.pt" ;  python3 audit_mirror.py --src audit-logs console-logs --dst "{DRIVE_LOGS}" --delete

### Package the audit materials → `inzva_mri-audit-materials.zip` on Drive
Assembles `task3/{code, data-split.md, logs}` from the repository and the Drive folder. The weights-only final model goes in; the ~110 MB intermediate checkpoints stay in `runs/` on Drive (listed by sha256 in `logs/export/checksums.sha256`) unless `--with-checkpoints` fits under 10 GB.

In [ ]:
!python3 {WORK}/mrixfields-analysis/audit/make_audit_package.py     --logs-dir "{DRIVE_LOGS}" --out "{DRIVE_LOGS}/inzva_mri-audit-materials.zip"
!ls -la "{DRIVE_LOGS}"/*.zip